## Optional: batch transform
If you want to run off-line predictions on a large dataset or don't need a real-time endpoint, you can use SageMaker [batch-transform](https://docs.aws.amazon.com/sagemaker/latest/dg/batch-transform.html). This section demonstrates how to use [SageMaker batch transform](https://docs.aws.amazon.com/sagemaker/latest/dg/batch-transform.html).

<div class="alert alert-info">Run this session only if you trained an estimator with the SageMaker built-in XGBoost in the section <b>Optional: train with SageMaker built-in algorithms</b>.
</div>

In [ ]:
transform_s3_url = f"s3://{bucket_name}/{bucket_prefix}/transform"
model_name = f"from-idea-to-prod-transform-{strftime('%d-%H-%M-%S', gmtime())}"

<div class="alert alert-info"> 💡 To create a transformer, use either <b>option 1</b> or <b>option 2</b>
</div>

### Option 1: create a batch transformer from the trained estimator
You can use [`EstimatorBase.transformer()`](https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html#sagemaker.estimator.EstimatorBase.transformer) to create a transformer for an estimator if you have a trained estimator. 

In [ ]:
try:
    trained_trainer = builtin_trainer
except NameError:
    print("You don't have a trained ModelTrainer. Run training first.")

trained_trainer

In [ ]:
from sagemaker.core.resources import TransformJob, Model
from sagemaker.core.shapes import TransformInput, TransformOutput, TransformResources, TransformS3DataSource, DataSource as TransformDataSource

# Get model name from the built-in trainer's model
# If we already built a model above, reuse it; otherwise build one
try:
    transform_model_name = builtin_model_name
except NameError:
    from sagemaker.serve import ModelBuilder
    mb = ModelBuilder(model=builtin_trainer, role_arn=sm_role, instance_type=train_instance_type)
    m = mb.build()
    transform_model_name = m.model_name

print(f"Using model: {transform_model_name}")

<div style="border: 4px solid coral; text-align: center; margin: auto;">
    <p style=" text-align: center; margin: auto;">Skip the Option 2 and go to the section <b>Run transform job</b>.
    </p>
</div>

### Option 2: load a model from a training job
Alternatively, you can load a model from a model artifact produced by a training job. You create a transformer with that model.

In [ ]:
if training_job_name is None:
    print("You don't have saved trained job name. Run SageMaker built-in algorithm training or use a trained estimator.")

In [ ]:
from sagemaker.core.resources import TrainingJob, Model
from sagemaker.core.shapes import ContainerDefinition

tj = TrainingJob.get(training_job_name=training_job_name)
model_data_url = tj.model_artifacts.s3_model_artifacts
algo_spec = tj.algorithm_specification

sm_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=algo_spec.training_image,
        model_data_url=model_data_url,
    ),
    execution_role_arn=sm_role,
)

In [ ]:
# TransformJob will be created in the 'Run transform job' section below
transform_model_name = model_name

In [ ]:
!aws s3 cp ./tmp/test_x.csv {test_s3_url}/test_x.csv

In [ ]:
test_s3_url

### Run transform job



In [ ]:
from sagemaker.core.resources import TransformJob
from sagemaker.core.shapes import (
    TransformInput, TransformOutput, TransformResources,
    TransformS3DataSource, TransformDataSource
)

transform_job_name = f"from-idea-to-prod-transform-{strftime('%d-%H-%M-%S', gmtime())}"

transform_job = TransformJob.create(
    transform_job_name=transform_job_name,
    model_name=transform_model_name,
    transform_input=TransformInput(
        data_source=TransformDataSource(
            s3_data_source=TransformS3DataSource(
                s3_data_type='S3Prefix',
                s3_uri=f'{test_s3_url}/test_x.csv',
            )
        ),
        content_type='text/csv',
        split_type='Line',
    ),
    transform_output=TransformOutput(
        s3_output_path=transform_s3_url,
        accept='text/csv',
        assemble_with='Line',
    ),
    transform_resources=TransformResources(
        instance_type=train_instance_type,
        instance_count=1,
    ),
)

transform_job.wait()

In [ ]:
# Transform job already waited above, check status
print(f"Transform job {transform_job_name} completed.")

In [ ]:
transform_s3_url

### Evaluate predictions

In [ ]:
!aws s3 ls {transform_s3_url}/

In [ ]:
!aws s3 cp {transform_s3_url}/test_x.csv.out tmp/predictions.csv
!aws s3 cp $test_s3_url/test_y.csv tmp/test_y.csv

In [ ]:
predictions = pd.read_csv("tmp/predictions.csv", names=["y_prob"])
test_y = pd.read_csv("tmp/test_y.csv", names=['y'])

#### Crosstab

In [ ]:
pd.crosstab(
    index=test_y['y'].values,
    columns=np.array(np.round(predictions), dtype=float).squeeze(), 
    rownames=['actuals'], 
    colnames=['predictions']
)

In [ ]:
test_auc = roc_auc_score(test_y, predictions)
print(f"Test-auc: {test_auc:.4f}")

#### ROC curve

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(test_y, predictions)
roc_auc = metrics.auc(fpr, tpr)
roc_display = metrics.RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc,estimator_name='Holdout/Test Data - ROC curve')
roc_display.plot()
plt.show()

#### Confusion matrix

In [ ]:
class_names = ["no", "yes"]
confusion_matrix = metrics.confusion_matrix(test_y, np.round(predictions))
ax, fig = plot_confusion_matrix(confusion_matrix, class_names)

#### Precision-recall curve

In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import PrecisionRecallDisplay

prec, recall, _ = precision_recall_curve(test_y, predictions)
average_precision= metrics.average_precision_score(test_y, predictions)
pr_display = PrecisionRecallDisplay(precision=prec, recall=recall, average_precision=average_precision, estimator_name='Holdout/Test Data - AUPRC curve')
pr_display.plot()
plt.show()